In [1]:
!pip install gradio tensorflow scikit-learn opencv-python scikit-image pillow pandas matplotlib seaborn -q

import os
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import gradio as gr
from PIL import Image
import shutil
print("Setup complete!")

Setup complete!


In [2]:
from google.colab import files

# Create directories
os.makedirs('dataset/train', exist_ok=True)
os.makedirs('dataset/test', exist_ok=True)
os.makedirs('uploads', exist_ok=True)

print("Upload your dataset (zipped or individual images). Recommended: Kaggle Minerals or core photos.")
uploaded = files.upload()

# Example: If zipped
import zipfile
for fn in uploaded.keys():
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn, 'r') as zip_ref:
            zip_ref.extractall('dataset')
print("Dataset uploaded! Organize into subfolders like dataset/train/quartz/, dataset/train/pyrite/ etc.")

Upload your dataset (zipped or individual images). Recommended: Kaggle Minerals or core photos.


Saving DCID-7.jpg to DCID-7.jpg
Dataset uploaded! Organize into subfolders like dataset/train/quartz/, dataset/train/pyrite/ etc.


In [3]:
def preprocess_image(img_path, target_size=(224, 224)):
    img = cv2.imread(img_path)
    img = cv2.resize(img, target_size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Basic enhancements
    img = cv2.GaussianBlur(img, (5,5), 0)
    return img / 255.0

def extract_features(img):
    # Color features
    hsv = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2HSV)
    color_hist = cv2.calcHist([hsv], [0], None, [8], [0, 256]).flatten()

    # Texture (Haralick or simple GLCM via skimage)
    from skimage.feature import graycomatrix, graycoprops
    gray = cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    glcm = graycomatrix(gray, [1], [0], 256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0,0]

    return np.concatenate([color_hist, [contrast]])

print("Feature functions ready.")

Feature functions ready.


In [4]:
# Load sample images for clustering demo

image_dir = 'dataset'
image_paths = []

# Recursively find all image files in the dataset directory
for root, _, files in os.walk(image_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            image_paths.append(os.path.join(root, file))


if not image_paths:
    print(f"Warning: No image files found in '{image_dir}'. Please ensure your dataset is correctly organized.")

    if os.path.exists('DCID-7.jpg'):
        image_paths = ['DCID-7.jpg']
        print("Using 'DCID-7.jpg' as a fallback since no dataset images were found.")
    else:
        raise FileNotFoundError(f"No image files found in '{image_dir}' and 'DCID-7.jpg' is also not present.")


features = []

for p in image_paths[:100]:
    img = preprocess_image(p)
    feat = extract_features(img)
    features.append(feat)

features = np.array(features)

# Dynamically set n_clusters to be at most the number of samples, and at least 1
num_samples = len(features)
if num_samples == 0:
    print("No features extracted for clustering. KMeans will not be run.")
    kmeans = None
else:
    n_clusters_to_use = min(num_samples, 5) # Use at most 5 clusters, or fewer if not enough samples
    kmeans = KMeans(n_clusters=n_clusters_to_use, random_state=42, n_init='auto').fit(features)  # e.g., 5 mineral groups
    print(f"Unsupervised clusters ready ({n_clusters_to_use} clusters). Use for discovering similar samples.")

Using 'DCID-7.jpg' as a fallback since no dataset images were found.
Unsupervised clusters ready (1 clusters). Use for discovering similar samples.


In [5]:

num_classes = 10

base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(128, activation='relu')(x)
output = Dense(num_classes, activation='softmax')(x)  # num_classes = your mineral count
model = Model(base_model.input, output)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])



94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [6]:
mineral_db = {
    'quartz': {'color': 'White/gray', 'hardness': 7, 'texture': 'Glassy, conchoidal fracture', 'common_in': 'Veins, host rock'},
    'pyrite': {'color': 'Brassy yellow', 'hardness': 6-6.5, 'texture': 'Cubic crystals', 'common_in': 'Sulfides'},
    # Add more: biotite, chalcopyrite, etc.
}

def identify_mineral(image):
    img_proc = preprocess_image(image) if isinstance(image, str) else image
    # Use your best model (CNN preferred)
    # pred = model.predict(np.expand_dims(img_proc, 0))
    # mineral = class_names[np.argmax(pred)]
    mineral = 'quartz'  # placeholder
    feats = mineral_db.get(mineral, {})
    return mineral, feats, extract_features(img_proc)  # visual features

In [7]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
import io
from PIL import Image

# ====================== PIXEL HIGHLIGHTING FUNCTION ======================
def generate_mineral_heatmap(original_image, model, class_idx):
    """
    Generates heatmap showing which pixels the model focuses on for the predicted mineral.
    Works with CNN models.
    """
    # Ensure image is (1, 224, 224, 3)
    if len(original_image.shape) == 3:
        img_array = np.expand_dims(original_image, axis=0)
    else:
        img_array = original_image

    # Get the last convolutional layer
    last_conv_layer = None
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            last_conv_layer = layer.name
            break

    if last_conv_layer is None:
        # Fallback: Simple color-based mask if no CNN
        gray = cv2.cvtColor((original_image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        _, mask = cv2.threshold(gray, 100, 255, cv2.THRESH_BINARY)
        heatmap = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
        return heatmap

    # Create Grad-CAM model
    grad_model = Model(
        inputs=model.input,
        outputs=[model.get_layer(last_conv_layer).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, class_idx]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_mean(tf.multiply(pooled_grads, conv_outputs), axis=-1)
    heatmap = np.maximum(heatmap, 0) / np.max(heatmap) if np.max(heatmap) != 0 else heatmap
    heatmap = cv2.resize(heatmap.numpy(), (original_image.shape[1], original_image.shape[0]))

    # Apply colormap
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    # Overlay on original
    overlay = cv2.addWeighted(
        (original_image * 255).astype(np.uint8), 0.6,
        heatmap, 0.4, 0
    )

    return overlay


# ====================== COMPARISON FUNCTION ======================
def create_comparison_visual(original_image, predicted_mineral, model, class_names):
    """
    Returns side-by-side comparison: Original | Highlighted Regions
    """
    try:
        class_idx = class_names.index(predicted_mineral)

        # Generate highlighted image
        highlighted = generate_mineral_heatmap(original_image, model, class_idx)

        # Create figure
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        axes[0].imshow(original_image)
        axes[0].set_title("Original Image")
        axes[0].axis('off')

        axes[1].imshow(highlighted)
        axes[1].set_title(f"Model Focus - {predicted_mineral} Regions")
        axes[1].axis('off')

        plt.suptitle(f"Pixel-wise Mineral Identification: {predicted_mineral}", fontsize=14)
        plt.tight_layout()

        # Convert to PIL Image for Gradio
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        buf.seek(0)
        comparison_img = Image.open(buf)

        plt.close(fig)
        return comparison_img

    except Exception as e:
        # Fallback simple visualization
        fallback = original_image.copy()
        cv2.putText(fallback, f"Predicted: {predicted_mineral}",
                   (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        return Image.fromarray(fallback.astype(np.uint8))

In [8]:
def process_upload(image):
    mineral, features, vis_feats = identify_mineral(image)
    logging = f"Mineral: {mineral}\nFeatures: {features}\nVisual: Contrast={vis_feats[-1]:.2f}"
    return mineral, str(features), logging

iface = gr.Interface(
    fn=process_upload,
    inputs=gr.Image(type="numpy"),
    outputs=[gr.Textbox(label="Predicted Mineral"),
             gr.Textbox(label="Mineral Properties"),
             gr.Textbox(label="Core Logging Summary")],
    title="Automatic Core Sample Logging & Mineral Identifier",
    description="Upload drill core / rock image for ML-based identification. Supports batch in extensions."
)

iface.launch(share=True)  # Shareable public link

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c337a0cf557a84263.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
